# Aspect-Wise Contrastive Embedding — Single GPU

Builds on `contrastive_training_starter.ipynb` and adds **aspect-wise
contrastive heads** (`ts_embed.loss.AspectContrastiveLoss`).

The encoder embedding is split into disjoint **chunks**, one per business
aspect. Each chunk has a private projection head and its own supervised-
contrastive loss with **aspect-specific positives**:

| chunk | dims | positive = customers sharing... |
|---|---|---|
| `spending` | `[0:96]` | MCC distribution, ticket size, velocity |
| `utilization` | `[96:192]` | utilization curve, paydown, revolve-vs-transact |

Because an aspect's loss only reads its own slice (and its head only takes
that slice as input), `d(loss_spending)/d(utilization dims) = 0` — each chunk
is optimized **only** by its own objective and specializes. The shared
transformer trunk still gets the summed gradient.

We define positives via **per-aspect KMeans clusters** over engineered
descriptors (same cluster = positive). The two masked/unmasked views of a
customer are always mutual positives too, so augmentation-invariance and
aspect-similarity are learned by the same SupCon term.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEmbeddingModel, TSEncoderConfig
from ts_embed.loss import AspectContrastiveLoss, AspectSpec, VICRegLoss, VICRegConfig

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data with two *independent* latent aspects

Feature layout (F_NUM=98): cols `0:50` = spend block, `50:98` = utilization
block. Two **independent** latent groups drive them:

- `spend_group` shapes the spend block (MCC mix / ticket size / velocity).
- `util_group` shapes the utilization block + the revolve categorical.

Independence is the point: a well-specialized `spending` chunk should recover
`spend_group` but **not** `util_group`, and vice versa.

In [ ]:
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
SP, UP = slice(0, 50), slice(50, 98)        # spend / utilization feature blocks
K_SPEND, K_UTIL = 3, 3                       # true number of latent groups

spend_group = np.random.randint(0, K_SPEND, N)
util_group  = np.random.randint(0, K_UTIL,  N)

numeric = 0.3 * np.random.randn(N, T, F_NUM).astype(np.float32)

# Spend block: each group has a distinct MCC-mix template + ticket-size scale
# + activity 'velocity' (temporal volatility).
spend_templates = np.random.randn(K_SPEND, F_NUM - 50).astype(np.float32) * 1.5
ticket_scale = np.array([0.7, 1.0, 1.6], np.float32)
velocity = np.array([0.1, 0.4, 0.9], np.float32)
for g in range(K_SPEND):
    m = spend_group == g
    base = spend_templates[g][None, None, :] * ticket_scale[g]
    noise = velocity[g] * np.random.randn(m.sum(), T, F_NUM - 50).astype(np.float32)
    numeric[m, :, SP] += base + noise

# Utilization block: group-specific utilization *curve* over the 24 months
# plus paydown slope.
t_axis = np.linspace(0, 1, T).astype(np.float32)
util_curves = np.stack([
    0.5 + 0.4 * np.sin(2 * np.pi * t_axis),      # revolver-ish, oscillating
    0.8 - 0.6 * t_axis,                          # steady paydown
    0.2 + 0.05 * t_axis,                         # transactor, low flat
]).astype(np.float32)
for g in range(K_UTIL):
    m = util_group == g
    curve = util_curves[g][None, :, None]        # (1,T,1)
    numeric[m, :, UP] += curve + 0.15 * np.random.randn(m.sum(), T, F_NUM - 50).astype(np.float32)

# Missingness, then 'impute' with per-feature mean.
missing = (np.random.rand(N, T, F_NUM) < 0.1).astype(np.uint8)
feat_mean = numeric.mean(axis=(0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)

# Categorical 0 = revolve(1) vs transact(0), tied to util_group; cat 1 = noise.
revolve_p = np.array([0.85, 0.5, 0.1], np.float32)[util_group]
categorical = np.stack([
    (np.random.rand(N, T) < revolve_p[:, None]).astype(np.int8),
    (np.random.rand(N, T) < 0.5).astype(np.int8),
], axis=-1)

data_dir = REPO_ROOT / 'data_demo'; data_dir.mkdir(exist_ok=True)
np.save(data_dir / 'numeric.npy', numeric)
np.save(data_dir / 'missing.npy', missing)
np.save(data_dir / 'categorical.npy', categorical)
print('saved', {p.name: np.load(p, mmap_mode='r').shape for p in sorted(data_dir.glob('*.npy'))})

## 2. Engineer aspect descriptors → per-aspect positive labels

In production you'd compute these from the real account history. Each aspect
gets a descriptor vector; KMeans on it yields a cluster id used as the
**positive label** for that aspect (same cluster ⇒ positive).

In [ ]:
def kmeans(X, k, iters=50, seed=0):
    """Tiny Lloyd KMeans (sklearn-free fallback); returns integer labels."""
    try:
        from sklearn.cluster import KMeans
        return KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    except ImportError:
        rng = np.random.default_rng(seed)
        C = X[rng.choice(len(X), k, replace=False)]
        for _ in range(iters):
            d = ((X[:, None, :] - C[None]) ** 2).sum(-1)
            lab = d.argmin(1)
            C = np.stack([X[lab == j].mean(0) if (lab == j).any() else C[j]
                          for j in range(k)])
        return lab

raw = np.load(data_dir / 'numeric.npy')
cat = np.load(data_dir / 'categorical.npy')

# Spending descriptor: MCC-mix (time-mean of spend feats, L1-normalized)
#                       + ticket size (overall magnitude)
#                       + velocity (mean |month-over-month change|).
spend_blk = raw[:, :, SP]
mcc_mix = spend_blk.mean(1)
mcc_mix = mcc_mix / (np.abs(mcc_mix).sum(1, keepdims=True) + 1e-6)
ticket = np.abs(spend_blk).mean((1, 2), keepdims=False)[:, None]
veloc = np.abs(np.diff(spend_blk, axis=1)).mean((1, 2))[:, None]
spend_desc = np.concatenate([mcc_mix, ticket, veloc], 1)

# Utilization descriptor: utilization curve (per-month mean of util feats)
#                          + paydown (mean negative MoM change)
#                          + revolve rate (categorical 0).
util_blk = raw[:, :, UP]
util_curve = util_blk.mean(2)                                  # (N,T)
paydown = np.clip(-np.diff(util_curve, axis=1), 0, None).mean(1)[:, None]
revolve = cat[:, :, 0].mean(1)[:, None]
util_desc = np.concatenate([util_curve, paydown, revolve], 1)

spending_label = kmeans(spend_desc, K_SPEND).astype(np.int64)
utilization_label = kmeans(util_desc, K_UTIL).astype(np.int64)

def cluster_recovery(pred, true):
    # best label permutation accuracy
    from itertools import permutations
    k = true.max() + 1
    return max(np.mean([p[t] == pr for t, pr in zip(true, pred)])
               for p in permutations(range(k)))
print(f'spending  cluster recovers spend_group : {cluster_recovery(spending_label, spend_group):.2f}')
print(f'utilization cluster recovers util_group: {cluster_recovery(utilization_label, util_group):.2f}')

## 3. Dataset + label-carrying collator

We wrap `TimeSeriesDataset` so each sample also yields its row index, then a
collator that (a) runs the existing `ContrastiveCollator` to build the two
masked/unmasked views and (b) attaches the per-aspect labels for that batch.

In [ ]:
paths = DatasetPaths(numeric=data_dir / 'numeric.npy',
                     missing=data_dir / 'missing.npy',
                     categorical=data_dir / 'categorical.npy')
base_ds = TimeSeriesDataset(paths)

class IndexedDataset(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]; item['_idx'] = i; return item

LABELS = {'spending': torch.from_numpy(spending_label),
          'utilization': torch.from_numpy(utilization_label)}

class AspectCollator:
    def __init__(self, masker):
        self.cc = ContrastiveCollator(masker)
    def __call__(self, batch):
        idx = torch.tensor([b.pop('_idx') for b in batch])
        out = self.cc(batch)
        out['labels'] = {k: v[idx] for k, v in LABELS.items()}
        return out

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30)
loader = DataLoader(IndexedDataset(base_ds), batch_size=256, shuffle=True,
                    drop_last=True, num_workers=0,
                    collate_fn=AspectCollator(masker))
print('batches/epoch:', len(loader))

## 4. Model + aspect heads (+ optional anti-collapse VICReg)

`d_model = 192`, split into `spending = [0:96]` and `utilization = [96:192]`.
SupCon's L2-normalization + many negatives already resist collapse; we add a
small VICReg variance/covariance term on the **full** embedding as cheap
insurance (set `VICREG_W = 0` to disable).

In [ ]:
D_MODEL = 192
cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=D_MODEL, n_layers=2, n_heads=4, proj_dim=128)
model = TSEmbeddingModel(cfg).to(device)

aspects = [
    AspectSpec('spending',    0,  96, proj_dim=64, temperature=0.1, weight=1.0),
    AspectSpec('utilization', 96, 192, proj_dim=64, temperature=0.1, weight=1.0),
]
aspect_loss = AspectContrastiveLoss(aspects).to(device)

VICREG_W = 0.1
vicreg = VICRegLoss(VICRegConfig(sim_coef=0.0, var_coef=1.0, cov_coef=0.04))

params = list(model.parameters()) + list(aspect_loss.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print('trainable params:', sum(p.numel() for p in params) / 1e6, 'M')

## 5. Training loop (single GPU)

We use the **encoder embedding** (`model.encode`, not the generic projector)
and feed it to the aspect loss. Watch each aspect's SupCon term fall
independently.

In [ ]:
EPOCHS = 12
hist = []
for epoch in range(EPOCHS):
    model.train(); aspect_loss.train()
    for batch in loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        emb_a = model.encode(num_a, mis_a, cat_a, keep_a)
        emb_b = model.encode(num_b, mis_b, cat_b, keep_b)

        al = aspect_loss(emb_a, emb_b, labels=labels)
        loss = al['loss']
        if VICREG_W > 0:
            vr = vicreg(emb_a, emb_b)
            loss = loss + VICREG_W * (vr['var'] + vr['cov'])

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()

    rec = {'epoch': epoch, 'total': float(al['loss']),
           'spending': float(al['spending']), 'utilization': float(al['utilization'])}
    hist.append(rec)
    print(f"epoch {epoch:2d}  total={rec['total']:.3f}  "
          f"spending={rec['spending']:.3f}  utilization={rec['utilization']:.3f}")

## 6. Specialization check

Extract encoder embeddings, then kNN-classify with **each chunk separately**:

- A specialized `spending` chunk → high accuracy on `spending_label`,
  near-chance on `utilization_label`.
- `utilization` chunk → the mirror image.

Also a per-chunk std collapse guard.

In [ ]:
model.eval()
embs = []
with torch.inference_mode():
    for i in range(0, N, 512):
        sl = slice(i, min(i + 512, N))
        n = torch.from_numpy(np.load(data_dir/'numeric.npy', mmap_mode='r')[sl].copy()).to(device)
        m = torch.from_numpy(np.load(data_dir/'missing.npy', mmap_mode='r')[sl].copy()).float().to(device)
        c = torch.from_numpy(np.load(data_dir/'categorical.npy', mmap_mode='r')[sl].astype(np.int64)).to(device)
        embs.append(model.encode(n, m, c).float().cpu().numpy())
embs = np.concatenate(embs)

def knn_acc(X, y, k=15, seed=0):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(X)); tr, te = perm[:4500], perm[4500:]
    Xtr = X[tr] / (np.linalg.norm(X[tr], axis=1, keepdims=True) + 1e-8)
    Xte = X[te] / (np.linalg.norm(X[te], axis=1, keepdims=True) + 1e-8)
    sims = Xte @ Xtr.T
    nn = np.argpartition(-sims, k, axis=1)[:, :k]
    from scipy.stats import mode
    try:
        pred = mode(y[tr][nn], axis=1, keepdims=False).mode
    except Exception:
        pred = np.array([np.bincount(row).argmax() for row in y[tr][nn]])
    return float((pred == y[te]).mean())

sp_chunk, ut_chunk = embs[:, 0:96], embs[:, 96:192]
print('                        spending_label   utilization_label')
print(f'spending chunk    ->    {knn_acc(sp_chunk, spending_label):.3f}            '
      f'{knn_acc(sp_chunk, utilization_label):.3f}')
print(f'utilization chunk ->    {knn_acc(ut_chunk, spending_label):.3f}            '
      f'{knn_acc(ut_chunk, utilization_label):.3f}')
print(f'chance             ->    {1.0/K_SPEND:.3f}            {1.0/K_UTIL:.3f}')

for nm, ch in [('spending', sp_chunk), ('utilization', ut_chunk)]:
    s = ch.std(0)
    print(f'{nm} chunk per-dim std: mean={s.mean():.3f} min={s.min():.3f}')
    assert s.mean() > 1e-2, f'COLLAPSE in {nm} chunk'
print('Expectation: diagonal cells high, off-diagonal near chance ='
      ' chunks specialized.')

## Next steps

- **Real positives**: replace KMeans labels with your production aspect
  descriptors. You can also pass `pos_masks={'spending': BxB bool}` to
  `aspect_loss(...)` instead of `labels=` to use a descriptor-kNN graph
  directly (e.g. cosine-similar spend profiles within a threshold).
- **More aspects**: add `AspectSpec` entries (e.g. `delinquency`, `tenure`);
  keep slices disjoint and summing to `d_model`. Tune `weight`/`temperature`
  per aspect.
- **Scale to cluster**: the loss is DDP-safe; move the SupCon batch wider by
  gathering features across ranks (mirror VICReg's `_maybe_all_gather`) if you
  need more negatives. Wire `labels` through the collator exactly as here and
  reuse `scripts/train_ddp.py`'s launch pattern.
- **If an off-diagonal stays high**: the two aspects' descriptors are
  correlated, the chunk is too large, or `weight` is imbalanced — shrink the
  chunk or down-weight the dominant aspect.